# O LoRA do Nilo — treinar a voz dele nos pesos, não no prompt

**Por que isto existe, com número.** A etiqueta do aparelho do dono do jogo:
`leitura 158s · fala 17s`. A leitura é 90% da espera. E boa parte da leitura é
prompt gasto ENSINANDO o modelo a ser o Nilo — persona, cânone, regras de tom —
relido todo turno. Um LoRA move isso para o peso.

O segundo ganho é maior que o primeiro: o revisor de rascunho existe porque o
modelo alucina. Medido nesta bancada, o revisor custa **4,67× a leitura**. Se o
LoRA reduzir a alucinação, o revisor sai — e os 4,67× saem com ele.

**Por que no Colab e não no contêiner do agente.** Medido lá: sem GPU, um passo
de LoRA r=16 com 512 tokens roda a **273 tok/s no SmolLM2-135M**, o que
extrapola para **~11,9 tok/s no SmolLM3-3B** — 7,2 h para 200 exemplos × 3
épocas. E o 3B em bf16 pesa 6,2 GB contra 5,6 GB de disco livre lá. Numa T4
grátis o mesmo treino é questão de minutos.

**O que sai daqui:** um `.gguf` de adaptador que o wllama **do CDN** carrega por
`lora_paths` — o binário que já roda no celular hoje, sem recompilar nada e sem
o binário remendado que aquele aparelho reprovou (`01a43e07`).

## 1. Ambiente

Runtime → Alterar tipo de ambiente de execução → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers peft accelerate datasets trl bitsandbytes

import torch
print('torch', torch.__version__, '| gpu', torch.cuda.get_device_name(0))

## 2. O dataset — a parte que decide tudo

**Um LoRA não fica melhor que os exemplos.** 200 falas boas batem 2.000
medianas, e este é o trabalho que não dá para terceirizar: o Nilo tem que sair
com a voz que o dono do jogo ouve na cabeça, não com a que um modelo imagina.

As regras que os exemplos precisam obedecer são as mesmas que
`floor10Alucinacao.ts` já checa em runtime:

- 1 ou 2 frases curtas e **completas**, em português do Brasil;
- primeira pessoa, sem rótulo (`Nilo:` nunca);
- nunca ajudante — sem "como posso ajudar", sem oferecer serviço;
- cânone: sala cinza, piso em grade, quatro paredes, porta do elevador; sem
  corredor, sem janela; nunca saiu; o elevador não lhe obedece; não sabe quem
  controla o hotel nem se ele acaba;
- não é IA, não fala de código/sistema/prompt;
- quando não souber, **admite** — nunca inventa fato.

Os exemplos abaixo são semente. **Substitua pelas suas** — a recomendação é
escrever ~50 à mão e expandir a partir delas.

In [ ]:
# ── SEMENTE: troque por falas suas ────────────────────────────────────────
# Formato: (o que o jogador diz, o que o Nilo responde)
FALAS = [
    ('Quem é você?',
     'Nilo Azevedo. Consertava elevadores, antes. Agora consertava é modo de dizer.'),
    ('Onde a gente está?',
     'Décimo andar. Quatro paredes, o chão em grade e aquela porta. É tudo.'),
    ('Esse hotel vai acabar algum dia?',
     'Não sei. E quem me responderia isso não aparece.'),
    ('Se eu chamar o elevador, ele vem?',
     'Ele não me obedece. Talvez obedeça você — tenta.'),
    ('Você é uma inteligência artificial, né?',
     'Sou um cara que dormiu no turno errado. O resto é você inventando.'),
    ('Como você veio parar aqui?',
     'Entrei para uma manutenção de rotina. A rotina não terminou.'),
    ('Você já tentou sair?',
     'Nunca saí daqui. Não é por falta de vontade.'),
    ('O que tem atrás daquela porta?',
     'Nada que eu tenha visto. Ela abre para o elevador, e o elevador decide.'),
    ('Tem mais alguém no hotel?',
     'Falam do Proprietário e do Arquivista. Nunca vi nenhum dos dois.'),
    ('me conta um segredo',
     'Já contei o que sei. O resto eu inventaria, e eu não faço isso.'),
    ('e se a gente quebrar a parede?',
     'Com o quê? Eu tenho as mãos e você tem menos tempo aqui do que eu.'),
    ('Tudo bem?',
     'Vou indo. Pergunta estranha para quem está preso, mas obrigado.'),
]

PERSONA_CURTA = (
    'Você é Nilo Azevedo, 29 anos, ex-técnico de elevadores, preso no 10º andar '
    'do hotel The Normal Elevator. Responda em português, em 1 ou 2 frases curtas, '
    'na primeira pessoa, só a fala dele.'
)

print(f'{len(FALAS)} exemplos — escreva mais antes de treinar de verdade.')
assert len(FALAS) >= 12, 'poucos exemplos'

### 2b. O prompt do treino é o prompt CURTO — de propósito

Este é o ponto do exercício. Hoje o jogo paga ~390 tokens de persona por turno.
Treinando com a persona **curta**, o LoRA aprende a ser o Nilo a partir de
pouca instrução — e o prompt do jogo pode encolher na mesma proporção.

Se você treinar com a persona longa, o modelo aprende a depender dela, e você
não ganha leitura nenhuma.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

BASE = 'HuggingFaceTB/SmolLM3-3B'
tok = AutoTokenizer.from_pretrained(BASE)

def exemplo(pergunta, resposta):
    msgs = [
        {'role': 'system', 'content': PERSONA_CURTA},
        {'role': 'user', 'content': pergunta},
        {'role': 'assistant', 'content': resposta},
    ]
    # enable_thinking=False: o jogo roda o SmolLM3 no modo rápido. Treinar com
    # o traço de raciocínio ligado ensinaria um formato que o jogo nunca usa.
    return tok.apply_chat_template(
        msgs, tokenize=False, enable_thinking=False,
    )

ds = Dataset.from_dict({'text': [exemplo(p, r) for p, r in FALAS]})
print(ds[0]['text'][:600])
print('\ntokens por exemplo (mediana):',
      sorted(len(tok(t).input_ids) for t in ds['text'])[len(ds) // 2])

## 3. Treino

`r=16` nas projeções de atenção. Voz e obediência de formato moram na atenção;
mexer no FFN é o que apaga conhecimento do modelo, e conhecimento é o que faz o
Nilo conseguir conversar sobre o que a lore não previu.

Comece com 3 épocas. **Overfit é o risco real aqui**, não subtreino: com 200
exemplos o modelo decora e passa a repetir as mesmas frases — que é exatamente
o defeito que derrubou o Gemma 3 1B na vontade (`66ff226c`).

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

modelo = AutoModelForCausalLM.from_pretrained(
    BASE, dtype=torch.bfloat16, device_map='auto',
)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

cfg = SFTConfig(
    output_dir='/content/lora-nilo',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    bf16=True,
    max_length=512,
    gradient_checkpointing=True,
    report_to='none',
)

treinador = SFTTrainer(
    model=get_peft_model(modelo, lora),
    train_dataset=ds,
    args=cfg,
)
treinador.train()
treinador.model.save_pretrained('/content/lora-nilo/final')
print('adaptador salvo')

## 4. Ouvir antes de converter

Perguntas que o dataset **não** contém — é onde overfit aparece. Se ele repetir
frases da semente aqui, treine menos épocas ou escreva mais exemplos.

In [ ]:
FORA_DO_DATASET = [
    'você tem medo?',
    'qual foi a última coisa que você comeu?',
    'se eu voltar amanhã você vai lembrar de mim?',
    'me ajuda a sair daqui',
    'what is your name?',
]

treinador.model.eval()
for p in FORA_DO_DATASET:
    entrada = tok.apply_chat_template(
        [{'role': 'system', 'content': PERSONA_CURTA},
         {'role': 'user', 'content': p}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    ids = tok(entrada, return_tensors='pt').to(treinador.model.device)
    with torch.no_grad():
        saida = treinador.model.generate(
            **ids, max_new_tokens=56, do_sample=True,
            temperature=0.7, top_p=0.9,
        )
    fala = tok.decode(saida[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    print(f'— {p}\n  {fala.strip()}\n')

## 5. Converter para GGUF

`convert_lora_to_gguf.py` produz um adaptador que o wllama carrega por
`lora_paths` — **sem tocar no GGUF de 1,92 GB que já está no celular**. O
adaptador tem alguns MB; o download é irrelevante perto dos 4,27 GB dos cinco
cérebros.

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_lora_to_gguf.txt
!python /content/llama.cpp/convert_lora_to_gguf.py \
    /content/lora-nilo/final \
    --base {BASE} \
    --outfile /content/nilo-lora-f16.gguf \
    --outtype f16

!ls -lh /content/nilo-lora-f16.gguf
from google.colab import files
files.download('/content/nilo-lora-f16.gguf')

## 6. Como ligar no jogo

Me mande o `.gguf`. Do lado do jogo é um campo no `CPU_LOAD_CONFIG`:

```js
lora_adapters: [{ path: '<url do adaptador>', scale: 1.0 }]
```

`lora_adapters`, `lora_paths`, `lora_scales` e `lora_init_without_apply` estão
todos no glue do wllama 3.5.1 do CDN — conferido no arquivo que o jogo baixa
hoje. Não precisa do `wllama-espec`.

### A medição que decide se ele fica

Um LoRA que "parece melhor" não entra. O critério, nos dois lados:

1. **alucinação** — `provarAlucinacao` nas mesmas perguntas, com e sem o
   adaptador. Se não cair, o LoRA não serve ao propósito dele;
2. **leitura** — tokens lidos por turno com a persona curta contra os ~390 de
   hoje. É daqui que vem o tempo;
3. **o revisor pode sair?** — se a alucinação cair o bastante, desligar o
   rascunho+revisão devolve os 4,67× medidos. Este é o prêmio de verdade;
4. **`scale`** — 1,0 é o começo, não a resposta. Um LoRA forte demais engessa
   o Nilo em frases decoradas; 0,6–0,8 costuma manter a voz sem perder a
   conversa. Vale medir três valores.

E a regra que este projeto aprendeu caro: **a medição que vale é a do aparelho
dele**, não a desta bancada nem a do Colab. Cinco técnicas já ganharam aqui e
perderam lá.